<a href="https://colab.research.google.com/github/Miaxrz/ML-Internship-Starter/blob/main/work/notebooks/w03_data_contract.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Miaxrz/ML-Internship-Starter/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

Unit of analysis: One row represents one anonymized content page (content_id) for a defined set of aggregated performance and content signals.

Time window: The main performance features such as impressions_90d, clicks_90d, and sessions_90d summarize the previous 90-day observation period. Additional fields compare shorter periods, including impressions_last_30d and impressions_prev_30d, with equivalent click and session fields. Content characteristics such as content_age_days and days_since_last_update describe the page's state at the time of the dataset snapshot.

For this Week 3 contract, I treat the decision point as the end of the observed feature window. The starter dataset does not provide a separate future outcome window, so trend_direction == "down" is treated as a proxy label derived from the observed window, not as a true future prediction target.

In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

import pandas as pd

# Load the starter dataset
df = pd.read_csv("/content/content_refresh_anonymized.csv")

print("Dataset shape:", df.shape)

print("\nTotal rows:", len(df))
print("Unique content IDs:", df["content_id"].nunique())

print("\nDuplicate content IDs:")
print(df["content_id"].duplicated().sum())

print("\nColumns related to time windows:")
time_columns = [
    col for col in df.columns
    if "90d" in col or "30d" in col or "days" in col
]

print(time_columns)

Dataset shape: (30000, 44)

Total rows: 30000
Unique content IDs: 30000

Duplicate content IDs:
0

Columns related to time windows:
['impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'days_with_impressions', 'days_with_sessions', 'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d', 'content_age_days', 'days_since_last_update']


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

**Features:** I will use observable signals that are available before the ranking decision. These include search demand, content characteristics, search performance, traffic, engagement, freshness, and page-position signals. Candidate features include:

* search_volume
* competition
* cpc
* word_count
* char_count
* impressions_90d
* clicks_90d
* sessions_90d
* users_90d
* engaged_sessions_90d
* ai_sessions_90d
* days_with_impressions
* days_with_sessions
* content_age_days
* days_since_last_update
* ctr
* avg_position
* engagement_rate
* scroll_rate
* ai_traffic_pct

I may also use safe categorical context about the page, such as content_type, main_intent, age_tier, freshness_tier, word_count_tier, impression_tier, and position_tier.


**Label / proxy:** For the starter dataset, I will use:

> trend_direction == "down"

This will be converted into a binary proxy label called is_declining_proxy. A value of 1 means the page is classified as having a downward trend in the observed dataset, while 0 means it is not classified as downward.

This is a proxy rather than an ideal target because it is derived from the current observation window rather than from a separate future period. Therefore, it will never be used as a feature.


**Context fields:** content_id and client_id will be used only for identification, grouping, deduplication, and validation. They will not be used as direct model features because the identifiers themselves do not represent meaningful numerical signals.

Additional fields such as provider_used and model_used will also be treated as context rather than predictive features unless a later analysis provides a justified, leakage-safe reason to use them.


**Excluded fields:** I will exclude any field that directly creates the proxy label from the model features. In particular, trend_direction will not be used as a feature because the label is calculated directly from it.

I will also exclude any future information or decision flags if they appear in later datasets, because information unavailable at the prediction point would create leakage. The internship dataset guide specifically warns against using product decision scores or flags as features or labels for discovery.

In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

# Define the data contract buckets

feature_columns = [
    "search_volume",
    "competition",
    "cpc",
    "word_count",
    "char_count",
    "impressions_90d",
    "clicks_90d",
    "pageviews_90d",
    "sessions_90d",
    "users_90d",
    "engaged_sessions_90d",
    "ai_sessions_90d",
    "scroll_events_90d",
    "days_with_impressions",
    "days_with_sessions",
    "content_age_days",
    "days_since_last_update",
    "ctr",
    "avg_position",
    "engagement_rate",
    "scroll_rate",
    "ai_traffic_pct"
]

context_columns = [
    "content_id",
    "client_id",
    "provider_used",
    "model_used"
]

label_source = "trend_direction"

excluded_columns = [
    "trend_direction"
]

# Check which planned fields exist in the dataset
print("FEATURE COLUMNS")
print([col for col in feature_columns if col in df.columns])

print("\nCONTEXT COLUMNS")
print([col for col in context_columns if col in df.columns])

print("\nLABEL SOURCE")
print(label_source, "exists:", label_source in df.columns)

print("\nEXCLUDED COLUMNS")
print([col for col in excluded_columns if col in df.columns])

# Verify that no excluded field appears in the feature list

overlap = set(feature_columns).intersection(excluded_columns)

print("\nFeature / excluded overlap:", overlap)

assert len(overlap) == 0, "Leakage risk: excluded column found in features"

print("✓ No excluded fields are included as model features.")

FEATURE COLUMNS
['search_volume', 'competition', 'cpc', 'word_count', 'char_count', 'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'days_with_impressions', 'days_with_sessions', 'content_age_days', 'days_since_last_update', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct']

CONTEXT COLUMNS
['content_id', 'client_id', 'provider_used', 'model_used']

LABEL SOURCE
trend_direction exists: True

EXCLUDED COLUMNS
['trend_direction']

Feature / excluded overlap: set()
✓ No excluded fields are included as model features.


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*


I verified the main claims in this data contract using direct dataframe checks. I checked the total row count, uniqueness of content_id, duplicate records, available time-window fields, missing values, missingness patterns by category, and the distribution of the proxy label. These checks are included below so that the data contract is based on measured properties of the dataset rather than assumptions.

In [6]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

# ============================================================
# 1. GRAIN CHECK
# ============================================================

print("GRAIN CHECK")
print("-" * 50)

print("Total rows:", len(df))
print("Unique content IDs:", df["content_id"].nunique())
print("Duplicate content IDs:", df["content_id"].duplicated().sum())


# ============================================================
# 2. MISSING VALUES CHECK
# ============================================================

print("\nMISSING VALUES")
print("-" * 50)

missing_summary = pd.DataFrame({
    "missing_count": df.isnull().sum(),
    "missing_percentage": (
        df.isnull().mean() * 100
    ).round(2)
})

missing_summary = missing_summary[
    missing_summary["missing_count"] > 0
].sort_values(
    "missing_percentage",
    ascending=False
)

display(missing_summary)


# ============================================================
# 3. MISSINGNESS BY CONTENT TYPE
# ============================================================

if "content_type" in df.columns:

    print("\nMISSINGNESS BY CONTENT TYPE")
    print("-" * 50)

    missing_by_content_type = (
        df.groupby("content_type")
        .apply(lambda x: x.isnull().mean() * 100)
        .round(2)
    )

    display(missing_by_content_type)


# ============================================================
# 4. PROXY LABEL CHECK
# ============================================================

df["is_declining_proxy"] = (
    df["trend_direction"]
    .astype(str)
    .str.lower()
    .eq("down")
    .astype(int)
)

print("\nPROXY LABEL DISTRIBUTION")
print("-" * 50)

print(df["is_declining_proxy"].value_counts())

print("\nDeclining proxy percentage:")
print(
    round(
        df["is_declining_proxy"].mean() * 100,
        2
    ),
    "%"
)


# ============================================================
# 5. TIME WINDOW FIELD CHECK
# ============================================================

print("\nTIME WINDOW FIELDS")
print("-" * 50)

time_columns = [
    col for col in df.columns
    if "90d" in col
    or "30d" in col
    or "days" in col
]

for col in time_columns:
    print("-", col)

GRAIN CHECK
--------------------------------------------------
Total rows: 30000
Unique content IDs: 30000
Duplicate content IDs: 0

MISSING VALUES
--------------------------------------------------


,missing_count,missing_percentage
provider_used,21438,71.46
word_count_tier,7699,25.66
char_count,7699,25.66
word_count,7699,25.66
char_count_tier,7699,25.66
model_used,5733,19.11
trend_pct,3388,11.29
competition_level,2610,8.70
search_volume,2468,8.23
competition,2468,8.23



MISSINGNESS BY CONTENT TYPE
--------------------------------------------------


/tmp/ipykernel_2383/285541710.py:51: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda x: x.isnull().mean() * 100)


,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,char_count_tier,ctr,avg_position,engagement_rate,scroll_rate,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct
content_type,,,,,,,,,,,,,,,,,,,,,
comparison article,0.0,0.0,0.00,0.00,0.00,0.00,0.0,0.00,0.0,0.0,...,0.0,0.0,0.0,0.0,0.29,0.0,0.0,0.0,0.0,0.29
feedly article,0.0,0.0,100.00,100.00,100.00,100.00,0.0,100.00,0.0,0.0,...,0.0,0.0,0.0,0.0,0.05,0.0,0.0,0.0,0.0,51.86
keyword article,0.0,0.0,1.37,1.37,1.89,1.37,0.0,1.02,28.3,28.3,...,28.3,0.0,0.0,0.0,0.45,0.0,0.0,0.0,0.0,8.45



PROXY LABEL DISTRIBUTION
--------------------------------------------------
is_declining_proxy
1    16262
0    13738
Name: count, dtype: int64

Declining proxy percentage:
54.21 %

TIME WINDOW FIELDS
--------------------------------------------------
- impressions_90d
- clicks_90d
- pageviews_90d
- sessions_90d
- users_90d
- engaged_sessions_90d
- ai_sessions_90d
- scroll_events_90d
- days_with_impressions
- days_with_sessions
- impressions_last_30d
- clicks_last_30d
- sessions_last_30d
- impressions_prev_30d
- clicks_prev_30d
- sessions_prev_30d
- content_age_days
- days_since_last_update


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

The starter dataset has several important limitations. First, it contains aggregated signals rather than a complete daily history, which limits the ability to study long-term patterns, seasonality, or exact changes over time. Second, the current proxy label `trend_direction == "down"` is derived from the observed data window, so it does not represent a true future outcome and may overlap conceptually with other performance signals. This creates a risk of circular reasoning if related trend-derived information is used without careful separation.

The dataset can show observed associations and support page prioritization, but it cannot prove that refreshing a page caused a performance improvement. It also cannot prove why a page declined, because factors such as seasonality, competition, search-result changes, or traffic consolidation may influence performance.

If I later use the full warehouse dataset, I must also account for its unbalanced client history and the fact that some early rows contain search data before GA4 tracking was available. Missing tracking data must not automatically be interpreted as zero traffic. I must also define non-overlapping feature and target windows to prevent leakage.

In [7]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

# Check important limitations in the starter dataset

key_columns = [
    "impressions_90d",
    "clicks_90d",
    "sessions_90d",
    "content_age_days",
    "trend_direction"
]

print("KEY FIELD MISSINGNESS")
print("-" * 50)

for col in key_columns:
    if col in df.columns:
        missing_count = df[col].isnull().sum()
        missing_pct = df[col].isnull().mean() * 100

        print(
            f"{col}: "
            f"{missing_count} missing "
            f"({missing_pct:.2f}%)"
        )


print("\nLIMITATION CHECK")
print("-" * 50)

print(
    "The starter dataset contains aggregated windows "
    "rather than a full daily time series."
)

print(
    "The decline label is a current-window proxy, "
    "not a future observed outcome."
)

KEY FIELD MISSINGNESS
--------------------------------------------------
impressions_90d: 0 missing (0.00%)
clicks_90d: 0 missing (0.00%)
sessions_90d: 0 missing (0.00%)
content_age_days: 0 missing (0.00%)
trend_direction: 0 missing (0.00%)

LIMITATION CHECK
--------------------------------------------------
The starter dataset contains aggregated windows rather than a full daily time series.
The decline label is a current-window proxy, not a future observed outcome.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.